# 10: Semantic Feature Extraction

Extracts location-agnostic semantic features from trajectories.
Downstream consumers: the recommender (notebook 16, `helpers/unified_recommender.py`) and the evaluation scripts.

## Feature Groups

| Group | Dims | Source |
|-------|------|--------|
| POI Category Distribution | 38 | Normalized visit proportions weighted by dwell time |
| Temporal Context | 12 | TimeOfDay, Weekpart, Season, duration metrics |
| Navigation Style | 4 | Casual, Exploratory, Immersive, Targeted |
| Engagement Metrics | 8 | POI count, diversity, dwell stats |
| Interaction Types | 5 | Passing, Meaningful, Immersive, Start, Destination |

Total: 67 dimensions.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from helpers.fuseki_config import query_endpoint
from helpers.semantic_features import FEATURE_GROUPS, SemanticFeatureExtractor

# TRAJECTO_FEATURES_DIR scopes the feature outputs per corpus, so a second
# ingestion does not overwrite the first corpus's feature matrix.
output_dir = Path(os.environ.get('TRAJECTO_FEATURES_DIR',
                                'output/semantic_features'))
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Endpoint: {query_endpoint}")
print(f"Feature groups: {FEATURE_GROUPS}")
print(f"Total dimensions: {sum(FEATURE_GROUPS.values())}")

## 1. Initialize Extractor

In [ ]:
extractor = SemanticFeatureExtractor()

trajectory_uris = extractor.get_all_trajectory_uris()
print(f"Trajectories: {len(trajectory_uris)}")
print(f"Feature dimensions: {len(extractor.feature_names)}")

## 2. Single Trajectory Inspection

Extract one trajectory to verify feature structure before batch processing.

In [ ]:
sample_uri = trajectory_uris[10]
sample_features = extractor.extract_trajectory_features(sample_uri)

feature_groups = extractor.get_feature_groups()
feature_names = extractor.get_feature_names()

print(f"Trajectory: {sample_uri.split('/')[-1]}")
print(f"Shape: {sample_features.shape}, non-zero: {np.count_nonzero(sample_features)}")
print()
for group, (start, end) in feature_groups.items():
    gf = sample_features[start:end]
    nz = np.count_nonzero(gf)
    if nz > 0:
        top_idx = np.argsort(gf)[::-1][:3]
        top = ', '.join(f"{feature_names[start+i]}={gf[i]:.3f}" for i in top_idx if gf[i] > 0)
        print(f"  {group} ({end-start}d): {nz} active — {top}")
    else:
        print(f"  {group} ({end-start}d): all zero")

## 3. Batch Extraction

Extract features for all trajectories. One SPARQL query per trajectory.

In [ ]:
all_features = extractor.batch_extract_features(trajectory_uris)

trajectory_ids = list(all_features.keys())
feature_matrix = np.array([all_features[uri] for uri in trajectory_ids])

print(f"Feature matrix: {feature_matrix.shape[0]} trajectories x {feature_matrix.shape[1]} dimensions")

## 4. Feature Statistics

Per-feature mean, std, and coverage. Sanity check before saving.

In [ ]:
feature_stats = pd.DataFrame({
    'feature': feature_names,
    'mean': np.mean(feature_matrix, axis=0),
    'std': np.std(feature_matrix, axis=0),
    'min': np.min(feature_matrix, axis=0),
    'max': np.max(feature_matrix, axis=0),
    'non_zero_pct': (feature_matrix > 0).mean(axis=0) * 100
})

# Top features by variance
feature_stats['variance'] = feature_stats['std'] ** 2
top = feature_stats.nlargest(15, 'variance')[['feature', 'mean', 'std', 'non_zero_pct']]
print("Top 15 features by variance:")
print(top.to_string(index=False))

## 5. Save Outputs

| File | Format | Consumer |
|------|--------|----------|
| `trajectory_features.json` | dict[uri → vector] | recommender, evaluation |
| `feature_metadata.json` | feature names + groups | recommender |
| `trajectory_ids.json` | ordered URI list | matrix index alignment |
| `feature_statistics.csv` | per-feature stats | reference |
| `feature_matrix.npy` | numpy array | direct loading |

In [ ]:
extractor.save_features(all_features, output_dir / 'trajectory_features.json')
extractor.save_metadata(output_dir / 'feature_metadata.json')

with open(output_dir / 'trajectory_ids.json', 'w') as f:
    json.dump(trajectory_ids, f, indent=2)

feature_stats.to_csv(output_dir / 'feature_statistics.csv', index=False)
np.save(output_dir / 'feature_matrix.npy', feature_matrix)

print(f"Saved {len(all_features)} trajectory feature vectors to {output_dir}/")